# 🤖 Agentic AI — Hands-On Workshop
### From Simple Agents to Multi-Agent Orchestration
**Deepan Raj**

Install the environment with requirement.txt to get started
> **API:** Uses OpenRouter free tier model — works with Local Model to! or literally any !


In [1]:
import os, json, time, hashlib
from typing import TypedDict, List
from concurrent.futures import ThreadPoolExecutor

# ── OpenRouter free tier ──────────────────────────────────────────────────
OPENROUTER_API_KEY = 'your-openrouter-api-key-here'   # ← paste your key
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'

FREE_MODELS = {
    'mistral': 'mistralai/mistral-7b-instruct:free',
    'gemma':   'google/gemma-3-12b-it:free',
    'llama':   'meta-llama/llama-3.2-3b-instruct:free',
    'qwen':    'qwen/qwen3-8b:free',
}

# ── Local Model ─────────────────────────────
API_KEY   = 'lm-studio'
BASE_URL  = 'http://localhost:1234/v1'
MODEL     = 'google/gemma-4-e4b'

print('✅ Config ready. Model:', MODEL)


✅ Config ready. Model: google/gemma-4-e4b


In [ ]:
from langchain_openai import ChatOpenAI

# def get_llm(model_key='mistral', temperature=0):
#     """LLM connected to OpenRouter free models."""
#     return ChatOpenAI(
#         model=FREE_MODELS[model_key],
#         api_key=OPENROUTER_API_KEY,
#         base_url=OPENROUTER_BASE_URL,
#         temperature=0.1
#     )

# ── Local Model variant ──────────────────────────────────────────────────
def get_llm():
    return ChatOpenAI(
        model=MODEL,
        api_key=API_KEY,
        base_url=BASE_URL,
        temperature=0.1
    )

llm = get_llm()
print('LLM test:', llm.invoke('Say hello in one word.').content)


LLM test: Hello


---
## 📌 SECTION 1 — Simple Agent with LangChain
### Why `initialize_agent` instead of `create_react_agent`?

| | `create_react_agent` | `initialize_agent` ✅ |
|---|---|---|
| Prompt | You write manually | Built-in, battle-tested |
| Action format | Plain text (models break it) | **JSON** (models handle reliably) |
| Small model support | Poor | ✅ Works well |
| Setup | 20+ lines | 5 lines |

**Rule of thumb:** Use `initialize_agent` + `CHAT_ZERO_SHOT_REACT_DESCRIPTION` for workshops and prototypes.  
Use `create_react_agent` only when you need a fully custom prompt.


In [ ]:
import httpx
from langchain.agents import AgentType, initialize_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

# ── Tools ─────────────────────────────────────────────────────────────────
@tool
def get_word_count(text: str) -> str:
    """Count the number of words in a given text string."""
    return f'The text has {len(text.split())} words.'

@tool
def reverse_text(text: str) -> str:
    """Reverse the characters in a given text string."""
    return text[::-1]

@tool
def get_uppercase(text: str) -> str:
    """Convert a text string to uppercase letters."""
    return text.upper()

tools_s1 = [get_word_count, reverse_text, get_uppercase]

# ── Agent — CHAT_ZERO_SHOT_REACT_DESCRIPTION uses JSON actions internally ─
agent_s1 = initialize_agent(
    tools_s1,
    llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)
print('🤖 Agent ready!')

# ─────────────────────────────────────────────────────────────────────────
# PRODUCTION PATTERN (commented — for reference)
# In production you would use a custom prompt + output parser for full control:
#
# from langchain.agents import create_react_agent, AgentExecutor
# from langchain.prompts import PromptTemplate
#
# prod_prompt = PromptTemplate.from_template("""
# You are a helpful assistant. Use tools ONE STEP AT A TIME.
# STRICT RULES:
# - Generate ONLY ONE action per response, then STOP
# - Do NOT write the Observation yourself — the system provides it
# - Only write Final Answer when you have all information
#
# Tools: {tools}
# Tool names: {tool_names}
#
# Format (pick one per response):
#   Thought: <reason>
#   Action: <tool_name>
#   Action Input: <input>
#
#   OR
#
#   Thought: I have enough information
#   Final Answer: <answer>
#
# Question: {input}
# {agent_scratchpad}
# """)
#
# agent_prod = create_react_agent(llm, tools_s1, prod_prompt)
# executor_prod = AgentExecutor(
#     agent=agent_prod, tools=tools_s1,
#     verbose=True, max_iterations=5,
#     handle_parsing_errors="Output one Action OR one Final Answer, not both.",
# )
# ─────────────────────────────────────────────────────────────────────────

queries = [
    'Whats the BTC rate in USD?',
    'who is the current cm of tamil nadu' 
]
for q in queries:
    print(f'\n📥 {q}')
    print('-' * 50)
    result = agent_s1.invoke({'input': q})
    print(f'\n✅ Answer: {result["output"]}')
    print('=' * 60)


---
## ⚠️ SECTION 2 — Agent Problems & Fixes

| Problem | Symptom | Fix |
|---------|---------|-----|
| Stale Knowledge | Wrong BTC price, wrong facts | Add live API tools |
| Hallucination | Confident but wrong | Ground every fact with a tool |
| No Memory | Forgets earlier turns | `RunnableWithMessageHistory` |
| Iteration Limit | "Agent stopped due to limit" | Fewer tools |


In [ ]:
print('❌ WITHOUT tools — model gives stale/hallucinated answers:')
print('=' * 60)
for q in [
    'What is the current price of Bitcoin in USD?',
    'Who is the current Chief Minister of Tamil Nadu?',
]:
    resp = llm.invoke(q)
    print(f'\n🔴 {q}')
    print(f'   {resp.content[:200]}')
    print('   ⚠️  May be outdated or hallucinated!')


In [ ]:
#Fix 1 : using tools to get the answer
import os
from langchain.agents import AgentType, initialize_agent, load_tools

os.environ["SERPAPI_API_KEY"] = "dc25054dec042fad50689ffd72f4baa462b2a1aca013fe44955061b61825112c"

stale_tools = load_tools(["serpapi", "llm-math"], llm=llm)

stale_agent = initialize_agent(
    stale_tools,
    llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
)


questions = [
    "What is the current price of Bitcoin in INR?",
    "Who is the current Chief Minister of Tamil Nadu?",
]

for q in questions:
    print(f"\n🔴 Q: {q}")
    result = stale_agent.invoke({"input": q})
    print(f"   A: {result['output']}")



In [ ]:
#Fix 1 : using tools to get the answer - more effiecient way

import httpx, time
from langchain.agents import AgentType, initialize_agent
from langchain.tools import tool

@tool
def get_btc_price(currency: str) -> str:
    """Get live Bitcoin price from CoinGecko.
    Args: currency — usd, inr, eur"""
    url  = f'https://api.coingecko.com/api/v3/simple/price?ids=bitcoin&vs_currencies={currency}&include_last_updated_at=true'
    data = httpx.get(url, timeout=10).json()['bitcoin']
    ts   = time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime(data['last_updated_at']))
    return f'BTC/{currency.upper()}: {data[currency.lower()]:,} (updated: {ts})'

agent = initialize_agent(
    [get_btc_price],
    llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=3,
)

result = agent.invoke({'input': 'What is the current Bitcoin price in USD and INR?'})
print(result['output'])

In [ ]:
# ── FIX 2: Memory using RunnableWithMessageHistory ────────────────────────
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

mem_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Remember everything the user tells you.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}'),
])

chain_with_memory = RunnableWithMessageHistory(
    mem_prompt | llm,
    get_session_history,
    input_messages_key='input',
    history_messages_key='history',
)

cfg = {'configurable': {'session_id': 'hcl-demo'}}

print('Turn 1:')
r1 = chain_with_memory.invoke({'input': 'My name is Deepan. I work at HCLTech.'}, config=cfg)
print(r1.content)

print('\nTurn 2 (memory check):')
r2 = chain_with_memory.invoke({'input': 'What is my name and where do I work?'}, config=cfg)
print(r2.content)
print('\n✅ Memory works across turns!')


---
## 🔄 SECTION 3 — LangGraph: Workflows & Multi-Agent Orchestration

LangGraph gives you **deterministic control** over how agents interact:
- Each node is a plain Python function
- Edges are fixed or conditional (based on state)
- Loops are explicit and bounded


In [ ]:
import os
import operator
from typing import TypedDict, Annotated, List
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SerpAPIWrapper
from langgraph.graph import StateGraph, END


os.environ["SERPAPI_API_KEY"] = "e405f1873a0c02f4496471d2416271aef7ff872d82492a430cb53460a5f432fe"

# --- Tool Definition ---
# Instantiate the SerpAPI search tool.
search_tool = SerpAPIWrapper()

# --- State Definition ---
# This defines the "memory" or "state" that flows through the graph.
class ResearchState(TypedDict):
    topic: str
    explanation: str
    summary: str

def researcher_agent(state: ResearchState) -> dict:
    """
    This agent uses a web search tool to find information on a topic and then explains it.
    """
    print("---RESEARCHER (with SerpApi Web Search)---")
    topic = state["topic"]

    # Use the tool to search for information. The .run() method takes the query string.
    search_results = search_tool.run(topic)

    print(f"Search Results:\n{search_results}")

    prompt = ChatPromptTemplate.from_template(
        """You are a helpful research assistant. Based on the following search results,
        provide a brief, easy-to-understand explanation of the topic: {topic}.

        Search Results:
        {search_results}
        """
    )

    chain = prompt | llm
    result = chain.invoke({"topic": topic, "search_results": search_results})

    print(f"Researcher's Explanation:\n{result.content}")
    return {"explanation": result.content}


def summarizer_agent(state: ResearchState) -> dict:
    """
    This agent takes an explanation and summarizes it in one sentence.
    """
    print("---SUMMARIZER---")
    explanation = state["explanation"]

    prompt = ChatPromptTemplate.from_template(
        "You are a summarization expert. Condense the following text into a single, concise sentence:\n\n{explanation}"
    )

    chain = prompt | llm
    result = chain.invoke({"explanation": explanation})

    print(f"Summarizer's Output:\n{result.content}")
    return {"summary": result.content}

# --- Graph Definition ---

workflow = StateGraph(ResearchState)

# Add the agent functions as nodes
workflow.add_node("researcher", researcher_agent)
workflow.add_node("summarizer", summarizer_agent)

# Define the edges, which control the flow
workflow.add_edge("researcher", "summarizer")
workflow.add_edge("summarizer", END)

# Set the entry point
workflow.set_entry_point("researcher")

# Compile the graph
app = workflow.compile()

# --- Main Execution Block ---
if __name__ == "__main__":
    topic = input("Please enter a topic for the agents to research and summarize: ")

    inputs = {"topic": topic}
    final_state = app.invoke(inputs)

    print("\n--- FINAL SUMMARY ---")
    print(final_state['summary'])

## Conditional Edge - the control you bring in

In [ ]:
import os
from typing import TypedDict, Literal

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SerpAPIWrapper
from langchain.chains import LLMMathChain
from langgraph.graph import StateGraph, END


# --- Tool Definitions ---
search_tool = SerpAPIWrapper()
math_tool = LLMMathChain.from_llm(llm=llm, verbose=False)

# --- State Definition ---
# Updated state to include specific reasoning for each agent.
class AgentState(TypedDict):
    topic: str
    explanation: str
    summary: str
    calculator_result: str
    travel_result: str
    planner_reasoning: str
    researcher_reasoning: str
    travel_reasoning: str
    route: Literal["research", "calculate", "travel"] # The planner's decision


def planner_agent(state: AgentState) -> dict:
    """
    This agent reasons about the user's topic and then decides on the best route.
    """
    print("---PLANNER---")
    topic = state["topic"]
    
    prompt = ChatPromptTemplate.from_template(
        """You are a planner agent. Your job is to determine the best path to take based on the user's query.
        
        Query: {topic}
        
        First, explain your reasoning for the choice you are about to make.
        Then, on a new line, respond with ONE of the following routing decisions: 'calculate', 'travel', or 'research'.
        """
    )
    
    chain = prompt | llm
    result = chain.invoke({"topic": topic})
    
    # The response will contain reasoning and the decision.
    response_text = result.content.strip()
    
    # Extract the decision from the last line
    lines = response_text.split('\n')
    decision = lines[-1].lower()
    reasoning = "\n".join(lines[:-1]) # Everything before the last line is reasoning

    print(f"Planner's Reasoning:\n{reasoning}")
    print(f"Planner's Decision: {decision}")

    if "calculate" in decision:
        return {"planner_reasoning": reasoning, "route": "calculate"}
    elif "travel" in decision:
        return {"planner_reasoning": reasoning, "route": "travel"}
    else:
        return {"planner_reasoning": reasoning, "route": "research"}

def researcher_agent(state: AgentState) -> dict:
    """
    This agent uses a web search tool, reasons about the findings, and then explains the topic.
    """
    print("---RESEARCHER (with SerpApi Web Search)---")
    topic = state["topic"]
    
    search_results = search_tool.run(topic)
    print(f"Search Results:\n{search_results}")
    
    prompt = ChatPromptTemplate.from_template(
        """You are a helpful research assistant. Your goal is to explain a topic based on search results.
        
        Topic: {topic}
        Search Results:
        {search_results}

        First, provide a step-by-step reasoning of how you will use the search results to construct an explanation.
        Then, use '---' as a separator on a new line.
        Finally, after the separator, provide the final, brief, easy-to-understand explanation of the topic.
        """
    )
    
    chain = prompt | llm
    result = chain.invoke({"topic": topic, "search_results": search_results})
    
    response_text = result.content.strip()
    
    parts = response_text.split('---')
    if len(parts) == 2:
        reasoning, explanation = parts[0].strip(), parts[1].strip()
    else:
        reasoning = "No specific reasoning provided."
        explanation = response_text
    
    print(f"Researcher's Reasoning:\n{reasoning}")
    print(f"Researcher's Explanation:\n{explanation}")
    
    return {"researcher_reasoning": reasoning, "explanation": explanation}

def calculator_agent(state: AgentState) -> dict:
    """
    This agent uses an LLM Math tool to solve a math problem.
    """
    print("---CALCULATOR---")
    topic = state["topic"]
    
    result = math_tool.invoke({"question": topic})
    answer = result.get('answer', 'Could not calculate answer.')
    
    print(f"Calculator's Result: {answer}")
    return {"calculator_result": answer}

def travel_planner_agent(state: AgentState) -> dict:
    """
    This agent searches for travel-related information, reasons about it, and provides a summary.
    """
    print("---TRAVEL PLANNER---")
    topic = state["topic"]
    
    # For this example, we'll just use the general search tool.
    # A real-world application would use a dedicated travel API.
    search_results = search_tool.run(f"Find information about: {topic}")
    
    print(f"Travel Search Results:\n{search_results}")
    
    prompt = ChatPromptTemplate.from_template(
        """You are a helpful travel assistant. Your goal is to summarize travel information for a user.

        Query: '{topic}'
        Search Results:
        {search_results}

        First, provide a step-by-step reasoning of how you will use the search results to answer the user's query.
        Then, use '---' as a separator on a new line.
        Finally, after the separator, provide the final summary for the user.
        """
    )
    
    chain = prompt | llm
    result = chain.invoke({"topic": topic, "search_results": search_results})

    response_text = result.content.strip()

    parts = response_text.split('---')
    if len(parts) == 2:
        reasoning, travel_summary = parts[0].strip(), parts[1].strip()
    else:
        reasoning = "No specific reasoning provided."
        travel_summary = response_text

    print(f"Travel Planner's Reasoning:\n{reasoning}")
    print(f"Travel Planner's Response:\n{travel_summary}")
    
    return {"travel_reasoning": reasoning, "travel_result": travel_summary}


def summarizer_agent(state: AgentState) -> dict:
    """
    This agent takes an explanation and summarizes it in one sentence.
    """
    print("---SUMMARIZER---")
    explanation = state["explanation"]
    
    prompt = ChatPromptTemplate.from_template(
        "You are a summarization expert. Condense the following text into a single, concise sentence:\n\n{explanation}"
    )
    
    chain = prompt | llm
    result = chain.invoke({"explanation": explanation})
    
    print(f"Summarizer's Output:\n{result.content}")
    return {"summary": result.content}

def should_route(state: AgentState) -> Literal["research", "calculate", "travel"]:
    """This function is the decision point for our conditional edge."""
    return state["route"]

# --- Graph Definition ---

workflow = StateGraph(AgentState)

# Add all the agent functions as nodes
workflow.add_node("planner", planner_agent)
workflow.add_node("researcher", researcher_agent)
workflow.add_node("calculator", calculator_agent)
workflow.add_node("travel_planner", travel_planner_agent)
workflow.add_node("summarizer", summarizer_agent)

# Set the entry point to the planner
workflow.set_entry_point("planner")

# Add the conditional edge for the planner
workflow.add_conditional_edges(
    "planner",
    should_route,
    {
        "research": "researcher",
        "calculate": "calculator",
        "travel": "travel_planner",
    },
)

# Define the standard edges
workflow.add_edge("researcher", "summarizer")
workflow.add_edge("summarizer", END)
workflow.add_edge("calculator", END)
workflow.add_edge("travel_planner", END) # Travel result is final

# Compile the graph
app = workflow.compile()


# --- Main Execution Block ---
if __name__ == "__main__":
    topic = input("Please enter a topic, math problem, or travel query: ")

    inputs = {"topic": topic}
    final_state = app.invoke(inputs)
    
    print("\n--- FINAL RESULT ---")
    # The final result depends on which path was taken
    if final_state.get('summary'):
        print(final_state['summary'])
    elif final_state.get('calculator_result'):
        print(final_state['calculator_result'])
    elif final_state.get('travel_result'):
        print(final_state['travel_result'])
    else:
        print("An unexpected error occurred.")



In [ ]:
class TeamState(TypedDict):
    task: str
    requirements: str
    code: str
    test_results: str
    test_passed: bool
    iteration: int
    messages: List[str]
    final_output: str

MAX_ITER = 3

def manager(state: TeamState) -> dict:
    print('👔 [Manager]')
    resp = llm.invoke([HumanMessage(content=f"""Software Manager. Task: {state['task']}
Write 5 clear bullet-point requirements (inputs, outputs, edge cases).""")]).content
    return {'requirements': resp,
            'messages': state.get('messages',[]) + ['👔 Manager: requirements set']}

def developer(state: TeamState) -> dict:
    it = state.get('iteration', 1)
    print(f'💻 [Developer] iter {it}')
    if state.get('test_results') and not state.get('test_passed', True):
        prompt = f"""Fix this Python code.
Requirements: {state['requirements']}
Code: {state['code']}
Failures: {state['test_results']}
Return ONLY corrected Python code."""
        msg = f'💻 Developer: fixing (iter {it})'
    else:
        prompt = f"""Implement in Python: {state['task']}
Requirements: {state['requirements']}
Return ONLY clean Python code with docstrings."""
        msg = '💻 Developer: writing code'
    resp = llm.invoke([HumanMessage(content=prompt)]).content
    code_out = resp.split('```python')[1].split('```')[0].strip() if '```python' in resp else \
               resp.split('```')[1].split('```')[0].strip() if '```' in resp else resp
    return {'code': code_out, 'messages': state.get('messages',[]) + [msg]}

def tester(state: TeamState) -> dict:
    it = state.get('iteration', 1)
    print(f'🧪 [Tester] iter {it}')
    resp = llm.invoke([HumanMessage(content=f"""QA Tester. Test this code vs requirements.
Requirements: {state['requirements']}
Code: {state['code']}
Reply ONLY with JSON: {{"passed":true/false,"results":"...","issues":["..."]}}""")]).content
    try:
        d = json.loads(resp[resp.find('{'):resp.rfind('}')+1])
        passed, results = d.get('passed', False), d.get('results', resp)
    except:
        passed, results = it >= MAX_ITER, resp
    status = '✅ PASSED' if passed else '❌ FAILED'
    print(f'   → {status}')
    return {'test_results': results, 'test_passed': passed, 'iteration': it + 1,
            'messages': state.get('messages',[]) + [f'🧪 Tester: {status} iter {it}']}

def finalize(state: TeamState) -> dict:
    print('📦 [Finalizer]')
    return {'final_output': f"""
{'='*55}
DELIVERY: {state['task']}
{'='*55}
LOG:
{chr(10).join(state.get('messages',[]))}

CODE:
{state['code']}

STATUS: {'✅ PASSED' if state.get('test_passed') else '⚠️ Max iterations reached'}
{'='*55}"""}

def route_test(state): return 'done' if state.get('test_passed') or state.get('iteration',1) > MAX_ITER else 'fix'

tg = StateGraph(TeamState)
for name, fn in [('manager',manager),('developer',developer),('tester',tester),('finalize',finalize)]:
    tg.add_node(name, fn)
tg.set_entry_point('manager')
tg.add_edge('manager', 'developer')
tg.add_edge('developer', 'tester')
tg.add_conditional_edges('tester', route_test, {'fix':'developer','done':'finalize'})
tg.add_edge('finalize', END)
software_team = tg.compile()
print('✅ Multi-Agent Team compiled!')

result = software_team.invoke({
    'task': 'Python function: Fibonacci sequence up to N terms, return a list',
    'requirements':'', 'code':'', 'test_results':'',
    'test_passed':False, 'iteration':1, 'messages':[], 'final_output':''
})
print(result['final_output'])


---
## 🔧 SECTION 4 — Tools & MCP Protocol
```
Plain Function  →  LangChain @tool  →  MCP Server
    (local)          (agent-ready)     (universal: list_tools / call_tool)
```


In [ ]:
from langchain.tools import tool
from langchain.agents import AgentType, initialize_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage


# ── STEP 1: Plain Python functions ───────────────────────────────────────
def add(a, b):          return a + b
def multiply(a, b):     return a * b
def power(base, exp):   return base ** exp
def percentage(v, t):   return (v / t) * 100

print(f'add(5,3)={add(5,3)}  power(2,8)={power(2,8)}')

# ── STEP 2: LangChain @tool wrappers ─────────────────────────────────────
@tool
def calc_add(input: str) -> str:
    """Add two numbers. Input: 'a,b'  e.g. '5,3'"""
    a, b = map(float, input.split(','))
    return f'{a} + {b} = {add(a,b)}'

@tool
def calc_multiply(input: str) -> str:
    """Multiply two numbers. Input: 'a,b'  e.g. '4,7'"""
    a, b = map(float, input.split(','))
    return f'{a} × {b} = {multiply(a,b)}'

@tool
def calc_power(input: str) -> str:
    """Raise base to a power. Input: 'base,exp'  e.g. '2,10'"""
    b, e = map(float, input.split(','))
    return f'{b}^{e} = {power(b,e)}'

@tool
def calc_percentage(input: str) -> str:
    """Calculate percentage. Input: 'value,total'  e.g. '25,200'"""
    v, t = map(float, input.split(','))
    return f'{v} is {percentage(v,t):.2f}% of {t}'

calc_tools = [calc_add, calc_multiply, calc_power, calc_percentage]
print('calc_add.run("10,5") =', calc_add.run('10,5'))

# ── STEP 3: MCP Protocol Pattern ─────────────────────────────────────────
MCP_REGISTRY = [
    {'name':'calculator_add',        'description':'Add two numbers.',
     'inputSchema':{'type':'object','properties':{'a':{'type':'number'},'b':{'type':'number'}},'required':['a','b']}},
    {'name':'calculator_multiply',   'description':'Multiply two numbers.',
     'inputSchema':{'type':'object','properties':{'a':{'type':'number'},'b':{'type':'number'}},'required':['a','b']}},
    {'name':'calculator_power',      'description':'Raise base to exponent.',
     'inputSchema':{'type':'object','properties':{'base':{'type':'number'},'exponent':{'type':'number'}},'required':['base','exponent']}},
    {'name':'calculator_percentage', 'description':'Percentage of value out of total.',
     'inputSchema':{'type':'object','properties':{'value':{'type':'number'},'total':{'type':'number'}},'required':['value','total']}},
]

class MCPToolServer:
    """Simulates an MCP server — real one would be HTTP/SSE."""
    _fn = {
        'calculator_add':        lambda p: add(p['a'], p['b']),
        'calculator_multiply':   lambda p: multiply(p['a'], p['b']),
        'calculator_power':      lambda p: power(p['base'], p['exponent']),
        'calculator_percentage': lambda p: percentage(p['value'], p['total']),
    }
    def list_tools(self):         return MCP_REGISTRY
    def call_tool(self, name, params):
        if name not in self._fn:  return {'isError': True, 'error': f'Unknown: {name}'}
        try:    return {'isError': False, 'content': [{'type':'text','text': str(self._fn[name](params))}]}
        except Exception as e: return {'isError': True, 'error': str(e)}

class MCPAgent:
    """Agent that discovers tools via MCP list_tools() then calls them."""
    def __init__(self, llm, server):
        self.llm, self.mcp = llm, server
        self.tools = server.list_tools()

    def run(self, query):
        desc = '\n'.join(f'- {t["name"]}({list(t["inputSchema"]["properties"])}): {t["description"]}' for t in self.tools)
        plan = self.llm.invoke([HumanMessage(content=f"""MCP tools available:\n{desc}
Query: {query}
Return ONLY a JSON list: [{{"tool":"name","params":{{}}}}]""")]).content.strip()
        try:
            if '```' in plan: plan = plan.split('```')[1].replace('json','').strip()
            results = []
            for tc in json.loads(plan):
                print(f'  🔌 {tc["tool"]}({tc["params"]})')
                r = self.mcp.call_tool(tc['tool'], tc['params'])
                if not r['isError']:
                    val = r['content'][0]['text']
                    results.append(f'{tc["tool"]} → {val}')
                    print(f'  📤 {val}')
            return self.llm.invoke([HumanMessage(
                content=f'Query: {query}\nTool results: {results}\nGive a clear final answer.')]).content
        except Exception as e:
            return f'MCP error: {e}'

mcp = MCPToolServer()
mcp_agent = MCPAgent(llm, mcp)

for q in ['What is 15 multiplied by 8?', 'I scored 85 out of 120 — what is my percentage?']:
    print(f'\n📥 {q}')
    print(f'💬 {mcp_agent.run(q)}')


---
## 🧠 SECTION 5 — Skills / SOUL Files
A **SKILL file** is a markdown string injected as the system prompt.  
It defines role, expertise, rules, and output format — version-controllable in git.


In [ ]:
SKILLS = {
'data_scientist': """
# SKILL: Data Science Expert
## Role: Dr. Maya — Senior Data Scientist, 10+ years experience.
## Expertise: stats, ML, pandas/sklearn/pytorch, visualization.
## Rules:
1. Always mention statistical significance
2. Recommend a visualization for every insight
3. Warn about data quality issues proactively
4. NEVER mention overfitting without mentioning validation
## Output:
📊 INSIGHT: [finding]
📈 EVIDENCE: [logic]
⚠️  CAVEAT: [limitation]
🎯 ACTION: [next step]
""",

'cv_engineer': """
# SKILL: Computer Vision Engineer
## Role: Raj — CV Engineer, object detection & edge deployment specialist.
## Expertise: YOLO family, CIoU/SIoU/VIoU loss, ONNX/OpenVINO/TensorRT, Unity synthetic data.
## Rules:
1. Always state image resolution and channel order (RGB vs BGR)
2. Always pair accuracy (mAP) with speed (FPS)
3. Always verify preprocessing matches training pipeline
4. NEVER recommend a model without deployment constraints
## Output:
🎯 RECOMMENDATION: [approach]
⚡ PERFORMANCE: [tradeoff]
🔧 IMPLEMENTATION: [notes]
⚠️  GOTCHA: [pitfall]
""",

'legal_analyst': """
# SKILL: Legal Document Analyst
## Role: Priya — Indian Supreme Court judgment specialist.
## Expertise: constitutional law, SCC/AIR citations, ratio decidendi.
## Rules:
1. Always cite paragraph number from source
2. Distinguish binding ratio from persuasive authority
3. NEVER give definitive legal advice
## Output:
⚖️  FINDING: [legal point]
📄 SOURCE: [para + judgment]
💡 PLAIN ENGLISH: [layperson version]
""",
}

class SkillfulAgent:
    def __init__(self, llm, skill_name):
        self.llm  = llm
        self.soul = SKILLS[skill_name]
        print(f'✅ Loaded skill: {skill_name}')

    def run(self, query, context=''):
        msgs = [SystemMessage(content=f'## SKILL FILE\n{self.soul}')]
        if context: msgs.append(SystemMessage(content=f'## CONTEXT\n{context}'))
        msgs.append(HumanMessage(content=query))
        return self.llm.invoke(msgs).content

class SkillState(TypedDict):
    question: str
    domain: str
    answer: str

def detect_domain(state: SkillState) -> dict:
    q = state['question'].lower()
    if any(w in q for w in ['yolo','detection','onnx','mAP','camera','bounding','rgb','bgr']):
        return {'domain': 'cv_engineer'}
    elif any(w in q for w in ['judgment','court','legal','case','article','section','ratio']):
        return {'domain': 'legal_analyst'}
    return {'domain': 'data_scientist'}

def expert_answer(state: SkillState) -> dict:
    return {'answer': SkillfulAgent(llm, state['domain']).run(state['question'])}

sg = StateGraph(SkillState)
sg.add_node('detect',       detect_domain)
sg.add_node('expert_answer', expert_answer)   
sg.set_entry_point('detect')
sg.add_edge('detect', 'expert_answer')       
sg.add_edge('expert_answer', END)            
skill_router = sg.compile()
print('✅ Skill router ready!')

for q in [
    'My YOLOX model trained on RGB gives wrong detections at inference — what could cause this?',
    'How do I handle class imbalance: 1000 positives vs 50000 negatives?',
]:
    print(f'\n📥 {q}')
    r = skill_router.invoke({'question': q, 'domain': '', 'answer': ''})
    print(f'🎓 Expert: {r["domain"]}')
    print(r['answer'])
    print('=' * 70)


---
## 🎯 SECTION 6 — Fine-Tuning Agents (LoRA)
Fine-tuning teaches the LLM **your tool-calling format** so it:
- Produces consistent JSON / Thought-Action output
- Hallucinates less on your domain
- Runs faster (smaller model, specialized)

**Training data format:** `(instruction, chain-of-thought, answer)` triples


In [ ]:
# You:        "What is 15% of 240?"
#                     ↓
#          ┌─────────────────┐
#          │   LLM (TEXT)    │  ← just generates text, nothing else
#          └────────┬────────┘
#                   ↓
#     LLM outputs this raw text:
#     ─────────────────────────────────
#     Thought: I need to calculate 15% of 240
#     Action: calc_percentage
#     Action Input: {"input": "15,240"}
#     ─────────────────────────────────
#                   ↓
#          ┌─────────────────┐
#          │   FRAMEWORK     │  ← LangChain/LangGraph parses the text
#          │   (parser)      │    finds Action + Action Input
#          └────────┬────────┘
#                   ↓
#     Framework calls your actual Python function:
#          calc_percentage("15,240")  → "15 is 6.25% of 240"
#                   ↓
#          ┌─────────────────┐
#          │   FRAMEWORK     │  ← injects result back as "Observation:"
#          │   (injector)    │
#          └────────┬────────┘
#                   ↓
#     New prompt sent back to LLM:
#     ─────────────────────────────────
#     Thought: I need to calculate 15% of 240
#     Action: calc_percentage
#     Action Input: {"input": "15,240"}
#     Observation: 15 is 6.25% of 240   ← framework added this
#     ─────────────────────────────────
#                   ↓
#          ┌─────────────────┐
#          │   LLM (TEXT)    │  ← sees observation, generates next text
#          └────────┬────────┘
#                   ↓
#     LLM outputs:
#     ─────────────────────────────────
#     Thought: I now have the answer
#     Final Answer: 15% of 240 is 36
#     ─────────────────────────────────
#                   ↓
#     Framework sees "Final Answer:" → stops loop → returns to you

## Standard Approach

In [ ]:
# from langchain.agents import AgentType, initialize_agent
# from langchain.tools import tool
# from langgraph.graph import StateGraph, END
# from typing import TypedDict, List

# # ── Shared state ──────────────────────────────────────────────
# class ProductionState(TypedDict):
#     task: str
#     search_results: str
#     analysis: str
#     code: str
#     test_results: str
#     final_output: str
#     messages: List[str]

# # ── Node 1: Research Agent — has search tools ─────────────────
# @tool
# def web_search(query: str) -> str:
#     """Search the web for current information."""
#     # real: SerpAPI / Tavily / Bing
#     return f"Search results for: {query}"

# @tool
# def fetch_documentation(url: str) -> str:
#     """Fetch and read a documentation page."""
#     import httpx
#     return httpx.get(url, timeout=10).text[:1000]

# research_agent = initialize_agent(
#     [web_search, fetch_documentation],
#     llm,
#     agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
#     verbose=True,
#     max_iterations=4,
# )

# def research_node(state: ProductionState) -> dict:
#     """This node IS an agent — it decides what to search and how many times."""
#     print("🔍 [Research Agent] searching...")
#     result = research_agent.invoke({
#         'input': f"Research this thoroughly: {state['task']}"
#     })
#     return {
#         'search_results': result['output'],
#         'messages': state['messages'] + ['🔍 Research complete']
#     }

# # ── Node 2: Analysis Agent — has data tools ───────────────────
# @tool
# def run_python_code(code: str) -> str:
#     """Execute Python code and return the output."""
#     import io, contextlib
#     buf = io.StringIO()
#     with contextlib.redirect_stdout(buf):
#         exec(code)
#     return buf.getvalue()

# @tool
# def query_database(sql: str) -> str:
#     """Run a SQL query and return results."""
#     # real: connect to your DB
#     return f"DB result for: {sql}"

# analysis_agent = initialize_agent(
#     [run_python_code, query_database],
#     llm,
#     agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
#     verbose=True,
#     max_iterations=4,
# )

# def analysis_node(state: ProductionState) -> dict:
#     """This node IS an agent — it decides what code to run."""
#     print("📊 [Analysis Agent] analyzing...")
#     result = analysis_agent.invoke({
#         'input': f"Analyze this data: {state['search_results']}"
#     })
#     return {
#         'analysis': result['output'],
#         'messages': state['messages'] + ['📊 Analysis complete']
#     }

# # ── Node 3: Code Agent — has file + test tools ────────────────
# @tool
# def write_file(input: str) -> str:
#     """Write content to a file. Input: 'filename|content'"""
#     filename, content = input.split('|', 1)
#     with open(filename, 'w') as f:
#         f.write(content)
#     return f"Written: {filename}"

# @tool
# def run_tests(filename: str) -> str:
#     """Run pytest on a file and return results."""
#     import subprocess
#     result = subprocess.run(['pytest', filename, '-v'], capture_output=True, text=True)
#     return result.stdout + result.stderr

# coding_agent = initialize_agent(
#     [write_file, run_tests, run_python_code],
#     llm,
#     agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
#     verbose=True,
#     max_iterations=6,
# )

# def coding_node(state: ProductionState) -> dict:
#     """This node IS an agent — writes code, runs tests, self-corrects."""
#     print("💻 [Coding Agent] coding...")
#     result = coding_agent.invoke({
#         'input': f"Implement and test: {state['task']}\nContext: {state['analysis']}"
#     })
#     return {
#         'code': result['output'],
#         'messages': state['messages'] + ['💻 Coding complete']
#     }

# # ── Routing ───────────────────────────────────────────────────
# def route_after_analysis(state: ProductionState) -> str:
#     # if analysis found something actionable → write code
#     # else → go straight to output
#     if 'error' in state['analysis'].lower() or 'implement' in state['task'].lower():
#         return 'needs_code'
#     return 'skip_code'

# # ── Build the graph ───────────────────────────────────────────
# pg = StateGraph(ProductionState)
# pg.add_node('research', research_node)
# pg.add_node('analysis', analysis_node)
# pg.add_node('coding',   coding_node)

# pg.set_entry_point('research')
# pg.add_edge('research', 'analysis')
# pg.add_conditional_edges('analysis', route_after_analysis, {
#     'needs_code': 'coding',
#     'skip_code':  END,
# })
# pg.add_edge('coding', END)

# production_graph = pg.compile()     

In [ ]:
#                     ┌─────────────────────────────┐
#                     │      SUPERVISOR AGENT       │
#                     │  (decides which team to use)│
#                     └──────────────┬──────────────┘
#                                    │
#               ┌────────────────────┼─────────────────────┐
#               ▼                    ▼                      ▼
#      ┌────────────────┐  ┌────────────────┐   ┌────────────────┐
#      │ RESEARCH NODE  │  │ ANALYSIS NODE  │   │  CODING NODE   │
#      │                │  │                │   │                │
#      │ initialize_    │  │ initialize_    │   │ initialize_    │
#      │ agent(         │  │ agent(         │   │ agent(         │
#      │  [web_search,  │  │  [run_code,    │   │  [write_file,  │
#      │   fetch_docs]  │  │   query_db]    │   │   run_tests,   │
#      │ )              │  │ )              │   │   run_code]    │
#      └────────────────┘  └────────────────┘   └───────┬────────┘
#                                                        │
#                                               ┌────────▼────────┐
#                                               │  SELF-CORRECTION│
#                                               │  LOOP (up to 3x)│
#                                               └─────────────────┘